In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import joblib

In [ ]:
def extract_motion_features(df, game_id, gsis_play_id, nflId, start_frame, end_frame):
    """Build a fixed-length feature vector summarizing one player's motion
    segment (start_frame through end_frame, inclusive of both), for use as
    ML model input. Returns a dict, or None if the play/segment can't be
    found. `df` is expected to be motion_tracking_data (or anything with the
    same columns): game_id, gsis_play_id, nflId, frame_id, x, y, s, a, dir,
    event, x_ball_at_snap, y_ball_at_snap, play_direction, position_group,
    teamId, qbAlignment."""
    play_df = df[(df['game_id'] == game_id) & (df['gsis_play_id'] == gsis_play_id)]
    if play_df.empty:
        return None

    ball_x = play_df['x_ball_at_snap'].iloc[0]
    ball_y = play_df['y_ball_at_snap'].iloc[0]
    direction_mult = -1 if play_df['play_direction'].iloc[0] == 'left' else 1

    qb_rows = play_df[play_df['position_group'] == 'QB']
    qb_x = qb_y = None
    if not qb_rows.empty and 'ball_snap' in qb_rows['event'].values: 
        qb_snap = qb_rows[qb_rows['event'] == 'ball_snap'].iloc[-1]
        qb_x, qb_y = qb_snap['x'], qb_snap['y']

    snap_rows = play_df[play_df['event'] == 'ball_snap']
    snap_frame = snap_rows['frame_id'].iloc[0] if len(snap_rows) else None

    p_df = play_df[play_df['nflId'] == nflId].sort_values('frame_id').reset_index(drop=True)
    if p_df.empty:
        return None

    start_matches = p_df.index[p_df['frame_id'] == start_frame]
    end_matches = p_df.index[p_df['frame_id'] == end_frame]
    if len(start_matches) == 0 or len(end_matches) == 0:
        return None

    seg = p_df.iloc[start_matches[0]:end_matches[0] + 1].reset_index(drop=True)
    if len(seg) < 2:
        return None

    dx = seg['x'].diff()
    dy = seg['y'].diff()
    dist_per_frame = np.sqrt(dx**2 + dy**2)
    total_distance = float(dist_per_frame.sum())

    start_x, start_y = seg['x'].iloc[0], seg['y'].iloc[0]
    end_x, end_y = seg['x'].iloc[-1], seg['y'].iloc[-1]
    net_displacement = float(np.hypot(end_x - start_x, end_y - start_y))

    dir_vals = seg['dir'].ffill().bfill().values
    turn_diffs = np.diff(dir_vals) if len(dir_vals) > 1 else np.array([0.0])
    turn_diffs = (turn_diffs + 180) % 360 - 180
    net_signed_turn = float(turn_diffs.sum())
    total_abs_turn = float(np.abs(turn_diffs).sum())

    start_y_rel_ball = start_y - ball_y
    end_y_rel_ball = end_y - ball_y
    crosses_ball_y = int(np.sign(start_y_rel_ball) != np.sign(end_y_rel_ball))

    depth_into_backfield = (seg['x'] - ball_x) * (-direction_mult)

    features = {
        'duration_frames': len(seg),
        'total_distance': total_distance,
        'net_displacement': net_displacement,
        'straightness': (net_displacement / total_distance) if total_distance > 0 else 0.0,
        'max_speed': float(seg['s'].max()),
        'avg_speed': float(seg['s'].mean()),
        'start_speed': float(seg['s'].iloc[0]),
        'end_speed': float(seg['s'].iloc[-1]),
        'max_accel': float(seg['a'].max()) if 'a' in seg.columns else np.nan,
        'net_signed_turn': net_signed_turn,
        'total_abs_turn': total_abs_turn,
        'start_x_rel_ball': (start_x - ball_x) * direction_mult,
        'start_y_rel_ball': start_y_rel_ball,
        'end_x_rel_ball': (end_x - ball_x) * direction_mult,
        'end_y_rel_ball': end_y_rel_ball,
        'crosses_ball_y': crosses_ball_y,
        'max_depth_into_backfield': float(depth_into_backfield.max()),
        'position_group': p_df['position_group'].iloc[0] if 'position_group' in p_df.columns else None,
        'team_id': p_df['teamId'].iloc[0] if 'teamId' in p_df.columns else None,
        'qb_alignment': p_df['qbAlignment'].iloc[0] if 'qbAlignment' in p_df.columns else None,
    }

    if qb_x is not None:
        features['start_x_rel_qb'] = (start_x - qb_x) * (-direction_mult)
        features['start_y_rel_qb'] = start_y - qb_y
        features['end_x_rel_qb'] = (end_x - qb_x) * (-direction_mult)
        features['end_y_rel_qb'] = end_y - qb_y
        features['start_dist_from_qb'] = float(np.hypot(start_x - qb_x, start_y - qb_y))
        features['end_dist_from_qb'] = float(np.hypot(end_x - qb_x, end_y - qb_y))
    else:
        for k in ['start_x_rel_qb', 'start_y_rel_qb', 'end_x_rel_qb', 'end_y_rel_qb',
                  'start_dist_from_qb', 'end_dist_from_qb']:
            features[k] = np.nan

    if snap_frame is not None:
        snap_matches_p = p_df.index[p_df['frame_id'] == snap_frame]
        if len(snap_matches_p):
            features['frames_to_snap_start'] = int(start_matches[0] - snap_matches_p[0])
            features['frames_to_snap_end'] = int(end_matches[0] - snap_matches_p[0])
        else:
            features['frames_to_snap_start'] = np.nan
            features['frames_to_snap_end'] = np.nan
    else:
        features['frames_to_snap_start'] = np.nan
        features['frames_to_snap_end'] = np.nan

    return features

In [ ]:
# --- Generate a labeling template: every candidate motion across whatever
# games are currently loaded in motion_tracking_data. Hand-label the
# resulting CSV's `manual_label` column (leave rule_based_category as
# reference only -- don't just copy it, that would just re-encode the same
# rule-based mistakes rather than give the model anything new to learn from).
label_rows = []

unique_plays_ml = motion_tracking_data[['game_id', 'gsis_play_id']].drop_duplicates()
for _, row in unique_plays_ml.iterrows():
    game_id, gsis_play_id = row['game_id'], row['gsis_play_id']
    play_motions = classify_motions(motion_tracking_data, game_id, gsis_play_id)
    for (nflId, motion_index), motion_info in play_motions.items():
        label_rows.append({
            'game_id': game_id,
            'gsis_play_id': gsis_play_id,
            'nflId': nflId,
            'motion_index': motion_index,
            'player_name': motion_info.get('player_name'),
            'start_frame': motion_info['start_frame'],
            'end_frame': motion_info['end_frame'],
            'rule_based_category': motion_info['category'],
            'manual_label': '',  # fill this in by hand against film
        })

labeling_template = pd.DataFrame(label_rows)
labeling_template.to_csv('motion_labeling_template.csv', index=False)
print(f"Wrote {len(labeling_template)} candidate motions to motion_labeling_template.csv")
labeling_template.head()

In [ ]:
# --- Once you've hand-labeled motion_labeling_template.csv (filled in
# manual_label) and saved it as motion_labels.csv, load it back in here.
labels_df = pd.read_csv('motion_labels.csv', dtype={'game_id': str, 'gsis_play_id': str})
labels_df = labels_df[labels_df['manual_label'].notna() & (labels_df['manual_label'] != '')].reset_index(drop=True)
print(f"{len(labels_df)} labeled motions loaded")
labels_df['manual_label'].value_counts()

In [ ]:
# --- Extract features for every labeled motion ---
feature_rows = []
valid_label_idx = []

for i, row in labels_df.iterrows():
    feats = extract_motion_features(
        motion_tracking_data, row['game_id'], row['gsis_play_id'],
        int(row['nflId']), row['start_frame'], row['end_frame']
    )
    if feats is not None:
        feature_rows.append(feats)
        valid_label_idx.append(i)
    else:
        print(f"WARNING: could not extract features for {row['game_id']}, {row['gsis_play_id']}, nflId={row['nflId']}")

X = pd.DataFrame(feature_rows)
y = labels_df.loc[valid_label_idx, 'manual_label'].reset_index(drop=True)

categorical_cols = ['position_group', 'team_id', 'qb_alignment']
numeric_cols = [c for c in X.columns if c not in categorical_cols]

print(f"Feature matrix: {X.shape[0]} rows, {X.shape[1]} columns")
X.head()

In [ ]:
# --- Train/test split + preprocessing + model ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), numeric_cols),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]), categorical_cols),
])

rf_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=300, max_depth=None, min_samples_leaf=2,
        class_weight='balanced', random_state=42, n_jobs=-1,
    )),
])

rf_pipeline.fit(X_train, y_train)
print("Random Forest test accuracy:", rf_pipeline.score(X_test, y_test))

In [ ]:
# --- Evaluation ---
y_pred = rf_pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, xticks_rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# --- Feature importance ---
onehot_cols = rf_pipeline.named_steps['preprocess'].named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_cols)
all_feature_names = numeric_cols + list(onehot_cols)
importances = rf_pipeline.named_steps['model'].feature_importances_

importance_df = pd.DataFrame({'feature': all_feature_names, 'importance': importances}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(8, 10))
top_n = importance_df.head(25)
ax.barh(top_n['feature'][::-1], top_n['importance'][::-1])
ax.set_xlabel('Importance')
ax.set_title('Top 25 feature importances (Random Forest)')
plt.tight_layout()
plt.show()

importance_df.head(15)

In [ ]:
# --- Alternative: HistGradientBoostingClassifier (often stronger on tabular
# data, handles NaN natively so imputation on the numeric side is optional
# here -- categoricals still need encoding though) ---
hgb_pipeline = Pipeline([
    ('preprocess', ColumnTransformer([
        ('num', 'passthrough', numeric_cols),
        ('cat', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_cols),
    ])),
    ('model', HistGradientBoostingClassifier(
        max_iter=300, random_state=42,
    )),
])

hgb_pipeline.fit(X_train, y_train)
print("HistGradientBoosting test accuracy:", hgb_pipeline.score(X_test, y_test))
print(classification_report(y_test, hgb_pipeline.predict(X_test)))

In [ ]:
# --- Save whichever model performed better ---
best_pipeline = rf_pipeline  # swap to hgb_pipeline if it scored higher above
joblib.dump({
    'pipeline': best_pipeline,
    'numeric_cols': numeric_cols,
    'categorical_cols': categorical_cols,
}, 'motion_classifier.joblib')
print("Saved motion_classifier.joblib")

In [ ]:
# --- Apply the saved model to new/unlabeled motions (e.g. a newly loaded
# game's motion_tracking_data) without needing to hand-label it first ---
def classify_motions_ml(df, game_id, gsis_play_id, model_path='motion_classifier.joblib'):
    """Same shape of output as classify_motions() -- {(nflId, motion_index):
    {..., 'category': predicted_label}} -- but using the trained model
    instead of the rule cascade."""
    saved = joblib.load(model_path)
    pipeline = saved['pipeline']

    play_motions = classify_motions(df, game_id, gsis_play_id)  # reuse motion
    # start/end detection (man_in_motion tagging, window caps) from the rule
    # pipeline -- only the final category label is replaced by the model.
    results = {}
    for key, motion_info in play_motions.items():
        nflId, motion_index = key
        feats = extract_motion_features(df, game_id, gsis_play_id, nflId,
                                         motion_info['start_frame'], motion_info['end_frame'])
        if feats is None:
            results[key] = motion_info
            continue
        feats_df = pd.DataFrame([feats])
        predicted = pipeline.predict(feats_df)[0]
        results[key] = {**motion_info, 'category': predicted, 'rule_based_category': motion_info['category']}
    return results